## Japanese Tokenization


In [4]:
import json
import spacy
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from tqdm import tqdm  # Added tqdm

# Force GPU usage
spacy.prefer_gpu()
nlp = spacy.load("ja_ginza")

class JapaneseFeatureExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.contents = []
        self.load_data()
        
        # 1. Tokenizing corpus with a progress bar
        print("Tokenizing corpus for BM25/TF-IDF...")
        self.tokenized_docs = []
        # Using nlp.pipe for faster batch processing if the corpus is large
        for doc in tqdm(nlp.pipe(self.contents, batch_size=50), total=len(self.contents), desc="Tokenizing"):
            self.tokenized_docs.append([t.orth_ for t in doc])
            
        self.bm25 = BM25Okapi(self.tokenized_docs)
        
        # TF-IDF Setup
        self.vectorizer = TfidfVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=lambda x: x)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.tokenized_docs)

    def load_data(self):
        # Count lines first for accurate tqdm progress
        num_lines = sum(1 for _ in open(self.file_path, 'r', encoding='utf-8'))
        with open(self.file_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, total=num_lines, desc="Loading JSONL"):
                data = json.loads(line)
                self.contents.append(data['content'])

    def get_sentence_features(self, doc_idx):
        doc = nlp(self.contents[doc_idx])
        sentences = list(doc.sents)
        doc_features = []

        # Feature extraction loop (no tqdm here as it's usually per-document)
        for s_idx, sent in enumerate(sentences):
            pos = s_idx / len(sentences) if len(sentences) > 1 else 0
            
            # Length normalization (safety check for empty sentences)
            max_len = max([len(s.text) for s in sentences]) if sentences else 1
            length = len(sent.text) / max_len

            sent_tokens = [t.orth_ for t in sent]
            bm25_score = self.bm25.get_scores(sent_tokens)[doc_idx]
            
            dep_path_1 = max([len(list(t.ancestors)) for t in sent]) if len(sent) > 0 else 0
            dep_path_2 = len([t for t in sent if t.dep_ in ("ROOT", "advcl", "ccomp")])

            ne_count = len(sent.ents)
            conj_count = len([t for t in sent if t.tag_.startswith("接続詞")])
            aux_count = len([t for t in sent if "助" in t.tag_])

            doc_features.append({
                "sentence": sent.text,
                "features": [pos, length, 0, dep_path_1, dep_path_2, bm25_score, ne_count, conj_count, aux_count]
            })
        return doc_features

# Run Pipeline
extractor = JapaneseFeatureExtractor(r'data_jp\JP_data.jsonl')

# If you want to process the ENTIRE corpus and see progress:
all_features = []
for i in tqdm(range(len(extractor.contents)), desc="Extracting Features"):
    all_features.append(extractor.get_sentence_features(i))

Loading JSONL: 100%|██████████| 1304/1304 [00:00<00:00, 68921.11it/s]


Tokenizing corpus for BM25/TF-IDF...


Tokenizing: 100%|██████████| 1304/1304 [33:49<00:00,  1.56s/it] 
d:\conda\envs\scraper\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
Extracting Features: 100%|██████████| 1304/1304 [1:06:08<00:00,  3.04s/it]


In [10]:
import pandas as pd

# Flatten the nested list for table format
flattened_rows = []
for doc_idx, doc_data in enumerate(all_features):
    for sent_data in doc_data:
        flattened_rows.append({
            "doc_id": doc_idx,
            "sentence": sent_data["sentence"],
            "features": sent_data["features"]
        })

# Create DataFrame and save
df = pd.DataFrame(flattened_rows)
df.to_pickle('data_jp/JP_features.pkl')
print("Saved to Pickle!")

Saved to Pickle!


In [11]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ... [Your existing JapaneseFeatureExtractor class remains the same] ...

# 1. Run Pipeline
extractor = JapaneseFeatureExtractor(r'data_jp\JP_data.jsonl')

# 2. Extract features with progress bar
all_rows = []
for i in tqdm(range(len(extractor.contents)), desc="Extracting Features"):
    doc_features = extractor.get_sentence_features(i)
    # Add doc_id so you can group sentences back to documents later
    for entry in doc_features:
        all_rows.append({
            "doc_id": i,
            "sentence": entry["sentence"],
            "pos": entry["features"][0],
            "length_norm": entry["features"][1],
            "dep_path_depth": entry["features"][3],
            "dep_root_count": entry["features"][4],
            "bm25_score": entry["features"][5],
            "ne_count": entry["features"][6],
            "conj_count": entry["features"][7],
            "aux_count": entry["features"][8]
        })

# 3. Convert to DataFrame and Save
df = pd.DataFrame(all_rows)
output_path = 'extracted_features.parquet'
df.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)

print(f"Successfully saved {len(df)} sentences to {output_path}")

Loading JSONL: 100%|██████████| 1304/1304 [00:00<00:00, 34052.48it/s]


Tokenizing corpus for BM25/TF-IDF...


Tokenizing: 100%|██████████| 1304/1304 [42:32<00:00,  1.96s/it] 
d:\conda\envs\scraper\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
Extracting Features:   4%|▎         | 48/1304 [01:57<51:15,  2.45s/it]  


KeyboardInterrupt: 

In [13]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# 1. Flatten your data as we did before
flattened_data = []
for doc_idx, doc_sentences in enumerate(all_features):
    for entry in doc_sentences:
        row = {
            "doc_id": doc_idx,
            "sentence": entry["sentence"],
            "pos": entry["features"][0],
            "len_norm": entry["features"][1],
            "dep_depth": entry["features"][3],
            "dep_root_count": entry["features"][4],
            "bm25_score": entry["features"][5],
            "ne_count": entry["features"][6],
            "conj_count": entry["features"][7],
            "aux_count": entry["features"][8]
        }
        flattened_data.append(row)

# 2. Create the DataFrame
df = pd.DataFrame(flattened_data)

# 3. Use PyArrow directly to bypass the Pandas/Arrow registration error
table = pa.Table.from_pandas(df)
pq.write_table(table, 'JP_features_output.parquet')

print("Success! Data saved to JP_features_output.parquet")

Success! Data saved to JP_features_output.parquet


In [15]:
df.columns

Index(['doc_id', 'sentence', 'pos', 'len_norm', 'dep_depth', 'dep_root_count',
       'bm25_score', 'ne_count', 'conj_count', 'aux_count'],
      dtype='object')

In [1]:
import json

file_path = 'data_jp/jp_data_with_questions.jsonl'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        count = 0
        for line in f:
            if count >= 10:
                break
            
            data = json.loads(line)
            # Accessing the column based on your request
            content = data.get('related_questions', 'Column not found')
            
            print(f"{count + 1}: {content}")
            print("-" * 30)
            count += 1
            
except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

1: ['ECの課題解決に金融工学をどのように応用しているのか？', '徳之島の『共生ジャム』プロジェクトで直面した課題や成功事例は何か？', '都市養蜂プロジェクトの具体的な進捗や成果はどうなっているのか？', '武蔵野大学での学生起業家の支援体制について教えてください。', '令和8年度の後期協定留学生修了式でどのようなプログラムが用意されていたのか？']
------------------------------


## Test for searching


In [3]:
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# 1. Setup OpenAI Client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
def semantic_search_test(query_text, top_k=3):
    # Load your saved database
    print("Loading embedding database...")
    vectors = np.load('jp_vectors.npy')  # Shape: (N, 3072)
    with open('jp_metadata.json', 'r', encoding='utf-8') as f:
        metadata = json.load(f)

    # 2. Embed the User's Query (The "Semantic" part)
    query_response = client.embeddings.create(
        input=query_text,
        model="text-embedding-3-large"
    )
    query_vector = np.array(query_response.data[0].embedding)

    # 3. Semantic Similarity via Dot Product
    # Since OpenAI vectors are normalized, Dot Product = Cosine Similarity
    scores = np.dot(vectors, query_vector)

    # 4. Rank Results
    top_indices = np.argsort(scores)[::-1][:top_k]

    print(f"\n--- 武蔵野大学 Assistant: Semantic Search Results for '{query_text}' ---")
    
    retrieved_context = []
    for idx in top_indices:
        item = metadata[idx]
        print(f"\n[Similarity Score: {scores[idx]:.4f}]")
        print(f"URL: {item['url']}")
        print(f"Summary Snippet: {item['summary'][:150]}")
        
        retrieved_context.append(item)
    
    return retrieved_context

# --- TEST IT ---
# Try a natural Japanese question
results = semantic_search_test("武蔵野大学の教育方針について知りたいです")

Loading embedding database...

--- 武蔵野大学 Assistant: Semantic Search Results for '武蔵野大学の教育方針について知りたいです' ---

[Similarity Score: 0.6962]
URL: https://www.musashino-u.ac.jp/basic/initial_policy.html
Summary Snippet: ①2015年9月の国連サミットで採択された、2016～2030年の15年間での達成を目指すSustainable Development Goals（SDGs）を構成する17の目標について、その169のターゲットの概要理解も含め、高校レベルよりもより進んだ理解ができるようにします。

[Similarity Score: 0.6856]
URL: https://www.musashino-u.ac.jp/admission/faculty/policies.html
Summary Snippet: 文学部では、本学学則第二条に記す教育目的を踏まえ、現代の世界を生きていく上で必要な幅広い教養と人間の文化についての深く豊かな専門的知見及び広い視野を身につけ、それらを基盤とした思考力・表現力・創造力により、文化の継承と発展に寄与し、社会のあり方を問い直すことができる人材を育成することを目指しています

[Similarity Score: 0.6842]
URL: https://www.musashino-u.ac.jp/news/detail/20250101-5928.html
Summary Snippet: このプログラムには、①文理を超えて「AI×専門」のデジタル人材を輩出していくための18科目からなるサブメジャー情報科目群「AI活用エキスパート」、②本学のブランドステートメント「世界の幸せをカタチにする。」に基づいて、2030年に向けたSDGsの取り組みを推進するため、SDGsの理念を学び、自ら問題


## Test with LLMs



In [ ]:
def get_musashino_answer(user_query):
    # --- STEP 1: RETRIEVAL (Semantic Search) ---
    vectors = np.load('jp_vectors.npy')
    with open('jp_metadata.json', 'r', encoding='utf-8') as f:
        metadata = json.load(f)

    # Embed the query
    query_vector = np.array(client.embeddings.create(
        input=user_query,
        model="text-embedding-3-large"
    ).data[0].embedding)

    # Semantic similarity (Dot Product)
    scores = np.dot(vectors, query_vector)
    top_indices = np.argsort(scores)[::-1][:2]  # Get top 2 most relevant chunks

    # --- STEP 2: AUGMENTATION (Building the Prompt) ---
    reference_text = ""
    for idx in top_indices:
        item = metadata[idx]
        reference_text += f"\n【資料のURL】: {item['url']}\n"
        reference_text += f"【要約】: {item['summary']}\n"
        reference_text += f"【内容】: {item['chunk_text']}\n"
        reference_text += "---------------------------\n"

   
    # --- STEP 3: GENERATION (The Assistant) ---
    system_prompt = """あなたは武蔵野大学の事務局アシスタントです。
        以下の【参考資料】の情報のみを使用して、ユーザーの質問に回答してください。

        【指示】
        1. 回答は簡潔かつ丁寧な日本語（です・ます調）で行ってください。
        2. 回答の最後に、必ず引用した資料の「URL」を明記してください。
        3. もし資料に答えが含まれていない場合は、「提供された資料の中には該当する情報が見つかりませんでした」と回答してください。"""

    user_prompt = f"【参考資料】:\n{reference_text}\n\nユーザーの質問: {user_query}"

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

# --- TEST THE FULL SYSTEM ---
query = "武蔵野大学の有明キャンパスへの行き方を教えてください。"
print(f"質問: {query}\n")
print(get_musashino_answer(query))

質問: 武蔵野大学の有明キャンパスへの行き方を教えてください。

武蔵野大学の有明キャンパスへの行き方については、以下の情報があります。

有明キャンパスの住所は、〒135-8181 東京都江東区有明三丁目3番3号です。最寄り駅からのアクセス方法については、具体的な情報が提供されていませんが、キャンパスの所在地に関する情報は以下のURLで確認できます。

【参考資料のURL】: https://www.musashino-u.ac.jp/ariake


: 